# ⚙️ Notebook 02 — Preprocesamiento
**Pasos:** Eliminación de columnas irrelevantes → Encoding → Normalización → Split Train/Test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os, pickle

os.makedirs('data/processed', exist_ok=True)
print('Librerías cargadas ✔')

In [ ]:
df = pd.read_csv('data/raw/ai_student_impact_dataset.csv')
print(f'Shape original: {df.shape}')
df.head(3)

## 1. Eliminación de columnas irrelevantes

In [ ]:
# Student_ID es un identificador único, no aporta información predictiva
df = df.drop(columns=['Student_ID'])
print(f'Shape tras eliminar Student_ID: {df.shape}')

## 2. Manejo de valores nulos

In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum().to_string())
print(f'\nTotal nulos: {df.isnull().sum().sum()}')
print('\n✔ No hay valores nulos. No se requiere imputación.')

## 3. Label Encoding — Variables Categóricas

KNN trabaja con distancias numéricas, por lo que todas las variables deben ser numéricas.  
Usamos `LabelEncoder` para variables nominales y ordinales.

| Variable | Tipo | Estrategia |
|---|---|---|
| `Major_Category` | Nominal | LabelEncoder |
| `Year_of_Study` | Ordinal | Mapeo manual (Freshman→1 … Graduate→5) |
| `Primary_Use_Case` | Nominal | LabelEncoder |
| `Prompt_Engineering_Skill` | Ordinal | Mapeo manual (Beginner→1, Intermediate→2, Advanced→3) |
| `Institutional_Policy` | Nominal | LabelEncoder |
| `Paid_Subscription` | Booleano | Ya es 0/1 |
| `Burnout_Risk_Level` | Target ordinal | Mapeo (Low→0, Medium→1, High→2) |

In [ ]:
df_enc = df.copy()

# --- Ordinal manual: Year_of_Study ---
year_map = {'Freshman': 1, 'Sophomore': 2, 'Junior': 3, 'Senior': 4, 'Graduate': 5}
df_enc['Year_of_Study'] = df_enc['Year_of_Study'].map(year_map)
print('Year_of_Study:', sorted(df_enc['Year_of_Study'].unique()))

# --- Ordinal manual: Prompt_Engineering_Skill ---
skill_map = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
df_enc['Prompt_Engineering_Skill'] = df_enc['Prompt_Engineering_Skill'].map(skill_map)
print('Prompt_Engineering_Skill:', sorted(df_enc['Prompt_Engineering_Skill'].unique()))

# --- LabelEncoder para nominales ---
le_dict = {}
nominal_cols = ['Major_Category', 'Primary_Use_Case', 'Institutional_Policy']
for col in nominal_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    le_dict[col] = le
    print(f'{col}: {list(zip(le.classes_, le.transform(le.classes_)))}')

# --- Booleano: Paid_Subscription ---
df_enc['Paid_Subscription'] = df_enc['Paid_Subscription'].astype(int)

# --- Target ---
burnout_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_enc['Burnout_Risk_Level'] = df_enc['Burnout_Risk_Level'].map(burnout_map)
print('\nTarget (Burnout_Risk_Level):', sorted(df_enc['Burnout_Risk_Level'].unique()))

# Guardar encoders
with open('data/processed/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

print('\n✔ Encoding completado')
df_enc.dtypes

## 4. División Train / Test (80% / 20%)

In [ ]:
X = df_enc.drop(columns=['Burnout_Risk_Level'])
y = df_enc['Burnout_Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train distribución:\n{y_train.value_counts().sort_index()}')
print(f'y_test  distribución:\n{y_test.value_counts().sort_index()}')
print('\n✔ Stratify garantiza proporciones similares en train y test')

## 5. Normalización — StandardScaler

**Justificación crítica para KNN:**  
KNN calcula distancias Euclidianas entre instancias. Sin escalar, variables con rangos grandes (e.g. `Skill_Retention_Score`: 10–100) dominarán sobre variables con rangos pequeños (e.g. `Tool_Diversity`: 1–5).  
El `StandardScaler` transforma cada variable a media=0, desviación estándar=1:
$$z = \frac{x - \mu}{\sigma}$$
**Importante:** El scaler se ajusta SÓLO con X_train y se aplica a X_test (evitar data leakage).

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform en train
X_test_sc  = scaler.transform(X_test)        # solo transform en test

print('Medias post-scaling (train, primeras 5 cols):')
print(np.round(X_train_sc[:, :5].mean(axis=0), 4))
print('\nDesv. std post-scaling (train, primeras 5 cols):')
print(np.round(X_train_sc[:, :5].std(axis=0), 4))

# Guardar scaler
with open('data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('\n✔ Scaling completado — scaler guardado')

In [ ]:
# ── Visualización: antes vs después del scaling ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.DataFrame(X_train, columns=X.columns).iloc[:, :5].boxplot(ax=axes[0])
axes[0].set_title('ANTES del Scaling', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

pd.DataFrame(X_train_sc, columns=X.columns).iloc[:, :5].boxplot(ax=axes[1])
axes[1].set_title('DESPUÉS del Scaling (StandardScaler)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Efecto del StandardScaler en las primeras 5 variables', fontsize=13)
plt.tight_layout()
plt.savefig('docs/figs/scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Guardar datos procesados ──────────────────────────────────────────────
np.save('data/processed/X_train_sc.npy', X_train_sc)
np.save('data/processed/X_test_sc.npy',  X_test_sc)
np.save('data/processed/y_train.npy',    y_train.values)
np.save('data/processed/y_test.npy',     y_test.values)

print('Archivos guardados en data/processed/ ✔')
print('\nResumen pipeline:')
print(f'  • Columnas eliminadas: Student_ID')
print(f'  • Nulos imputados: 0 (no había nulos)')
print(f'  • Encoding: LabelEncoder (nominales) + Mapeo ordinal (Year, Skill, Burnout)')
print(f'  • Scaling: StandardScaler (fit solo en train)')
print(f'  • Split: 80% train ({X_train_sc.shape[0]} obs) / 20% test ({X_test_sc.shape[0]} obs)')